In [1]:
from pypdf import PdfReader

pdf_path = "documents/ppe_manual.pdf"

reader = PdfReader(pdf_path)

print("Total pages:", len(reader.pages))

Total pages: 46


In [2]:
all_pages=  []
for page_number,page in enumerate(reader.pages):
    text = page.extract_text()
    if text:
        all_pages.append({
            "page": page_number+1,
            "text": text
        })
print("Pages with extracted text:", len(all_pages))

Pages with extracted text: 46


In [3]:
print(all_pages[0]["text"][:2000])

OSHA 3151-02R 2023
Personal Protective 
Equipment



In [4]:
def split_page_text(page_text,page_number,chunk_size=1000,overlap = 200):
    chunks=[]
    start = 0
    while start<len(page_text):
        end = start + chunk_size
        chunk_text = page_text[start:end]
        chunks.append({
            "page":page_number,
            "text":chunk_text
        })
        start = end-overlap
    return chunks

In [5]:
all_chunks = []

for page_data in all_pages:
    page_chunks = split_page_text(
        page_text=page_data["text"],
        page_number=page_data["page"],
        chunk_size=1000,
        overlap=200
    )

    all_chunks.extend(page_chunks)

print("Total chunks:", len(all_chunks))

Total chunks: 112


In [6]:
print(all_chunks[0])

{'page': 1, 'text': 'OSHA 3151-02R 2023\nPersonal Protective \nEquipment\n'}


In [7]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
chunk_texts = [chunk["text"] for chunk in all_chunks]



d:\Enterprise\.venv312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4570.46it/s]


In [8]:
embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy = True,
    show_progress_bar = True
)
print("Number of chunks:", len(all_chunks))
print("Embeddings shape:", embeddings.shape)

Batches: 100%|██████████| 4/4 [00:05<00:00,  1.44s/it]

Number of chunks: 112
Embeddings shape: (112, 384)


In [9]:
question = "What personal protective equipment is required?"
question_embedding = embedding_model.encode(
    question,
    convert_to_numpy=True
)

In [10]:
from sklearn.metrics.pairwise import cosine_similarity
scores = cosine_similarity(
    [question_embedding],
    embeddings
)[0]


In [11]:
import numpy as np
top_k = 3
top_indices = np.argsort(scores)[::-1][:top_k]
for idx in top_indices:
    print("=" * 60)
    print("Score:", scores[idx])
    print("Page:", all_chunks[idx]["page"])
    print("Text:")
    print(all_chunks[idx]["text"])

Score: 0.6350436
Page: 5
Text:
 
rotations) is an administrative control.
When engineering, work practice, and administrative controls 
are not feasible or do not provide sufficient protection, 
employers must provide personal protective equipment (PPE) 
to their employees and ensure its use. Personal protective 
equipment, commonly referred to as “PPE”, is equipment 
worn to minimize exposure to a variety of hazards. Examples 
of PPE include such items as gloves, foot and eye protection, 
protective hearing devices (earplugs, muffs) hard hats, 
respirators and full body suits.
This guide will help both employers and employees do the 
following:
 ■ Understand the types of PPE.
 ■ Know the basics of conducting a “hazard assessment” of 
the workplace.
 ■ Select appropriate PPE for a variety of circumstances.
 ■ Understand what kind of training is needed in the proper 
use and care of PPE.
The information in this guide is general in nature and does 
not address all workplace hazards or PP

In [12]:
context = ""
for idx in top_indices:
    context += f"Page {all_chunks[idx]['page']}:\n"
    context += all_chunks[idx]["text"]+"\n\n"

In [13]:
prompt = f"""
You are answering questions using a provided document.

Read the context carefully and answer the question using information
found in the context.

The wording of the question does NOT need to exactly match the wording
in the context. Use the meaning of the text.

If relevant information is present, summarize it clearly.

Only if there is truly no relevant information in the context, say:
"I could not find this information in the document."

Context:
{context}

Question:
{question}

Answer:
"""

In [14]:
import chromadb

client = chromadb.PersistentClient(
    path="chroma_db"
)

collection = client.get_or_create_collection(
    name="ppe_docs"
)

In [15]:
ids = [f"chunk_{i}" for i in range(len(all_chunks))]

documents = [chunk["text"] for chunk in all_chunks]

metadatas = [
    {
        "page": chunk["page"],
        "source": "ppe_manual.pdf"
    }
    for chunk in all_chunks
]

collection.add(
    ids=ids,
    documents=documents,
    embeddings=embeddings.tolist(),
    metadatas=metadatas
)

In [16]:
question = "What personal protective equipment is required?"

question_embedding = embedding_model.encode(
    question,
    convert_to_numpy=True
)

results = collection.query(
    query_embeddings=[question_embedding.tolist()],
    n_results=3
)

for i in range(len(results["documents"][0])):
    print("=" * 60)
    print("Text:")
    print(results["documents"][0][i])

    print("\nMetadata:")
    print(results["metadatas"][0][i])

    print("\nDistance:")
    print(results["distances"][0][i])

Text:
 
rotations) is an administrative control.
When engineering, work practice, and administrative controls 
are not feasible or do not provide sufficient protection, 
employers must provide personal protective equipment (PPE) 
to their employees and ensure its use. Personal protective 
equipment, commonly referred to as “PPE”, is equipment 
worn to minimize exposure to a variety of hazards. Examples 
of PPE include such items as gloves, foot and eye protection, 
protective hearing devices (earplugs, muffs) hard hats, 
respirators and full body suits.
This guide will help both employers and employees do the 
following:
 ■ Understand the types of PPE.
 ■ Know the basics of conducting a “hazard assessment” of 
the workplace.
 ■ Select appropriate PPE for a variety of circumstances.
 ■ Understand what kind of training is needed in the proper 
use and care of PPE.
The information in this guide is general in nature and does 
not address all workplace hazards or PPE requirements. The 
info

In [17]:
import os
from pypdf import PdfReader

documents_folder = "documents"

all_pages = []

for filename in os.listdir(documents_folder):

    if filename.endswith(".pdf"):

        pdf_path = os.path.join(documents_folder, filename)

        reader = PdfReader(pdf_path)

        for page_number, page in enumerate(reader.pages):

            text = page.extract_text()

            if text:
                all_pages.append({
                    "source": filename,
                    "page": page_number + 1,
                    "text": text
                })

print("Total extracted pages:", len(all_pages))

Total extracted pages: 148


In [18]:
def split_page_text(page_text, page_number, source, chunk_size=1000, overlap=200):

    chunks = []
    start = 0

    while start < len(page_text):

        end = start + chunk_size
        chunk_text = page_text[start:end]

        chunks.append({
            "source": source,
            "page": page_number,
            "text": chunk_text
        })

        start = end - overlap

    return chunks

In [19]:
all_chunks = []

for page_data in all_pages:

    page_chunks = split_page_text(
        page_text=page_data["text"],
        page_number=page_data["page"],
        source=page_data["source"]
    )

    all_chunks.extend(page_chunks)

print("Total chunks:", len(all_chunks))

Total chunks: 469


In [20]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
chunk_texts = [chunk["text"] for chunk in all_chunks]
chunk_texts = [chunk["text"] for chunk in all_chunks]
embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    show_progress_bar=True
)
print("Number of chunks", len(all_chunks))
print("Embedding shape",embeddings.shape)

Batches: 100%|██████████| 15/15 [00:17<00:00,  1.20s/it]

Number of chunks 469
Embedding shape (469, 384)


In [21]:
ids = [f"chunk{i}" for i in range(len(all_chunks))]
documents = [chunk["text"] for chunk in all_chunks]
metadatas = [
    {
        "source" : chunk["source"],
        "page": chunk["page"]
    }
    for chunk in all_chunks
]

In [22]:
collection.add(
    ids = ids,
    documents=documents,
    embeddings=embeddings.tolist(),
    metadatas=metadatas
)

In [41]:
question = "What is lockout tagout used for?"
question_embedding = embedding_model.encode(
    question,
    convert_to_numpy=True
)
results = collection.query(
    query_embeddings=[question_embedding.tolist()],
    n_results=5
)
filtered_docs = []
filtered_meta = []

distance_threshold = 0.7

for text, meta, distance in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
):
    print("Distance:", distance)

    if distance <= distance_threshold:
        filtered_docs.append(text)
        filtered_meta.append(meta)

print("Chunks kept:", len(filtered_docs))

Distance: 0.5621534585952759
Distance: 0.6134334802627563
Distance: 0.6408509016036987
Distance: 0.6681177020072937
Distance: 0.673374593257904
Chunks kept: 5


In [42]:
for i in range(len(results["documents"][0])):
    print("=" * 60)

    print("Source:", results["metadatas"][0][i]["source"])
    print("Page:", results["metadatas"][0][i]["page"])
    print("Distance:", results["distances"][0][i])

    print("\nText:")
    print(results["documents"][0][i])

Source: lockout_tagout.pdf
Page: 19
Distance: 0.5621534585952759

Text:
 
used, 
What are the requirements for lockout/tagout devices? 13 
Source: lockout_tagout.pdf
Page: 22
Distance: 0.6134334802627563

Text:
 or tagout device protection between outgoing and 
incoming shifts to control hazardous energy.  When lockout 
or tagout devices remain on energy-isolation devices from 
a previous shift, the incoming shift members must verify 
for themselves that the machinery is effectively isolated 
and deenergized. 
Control of Hazardous Energy (Lockout/Tagout) 16 
Source: lockout_tagout.pdf
Page: 15
Distance: 0.6408509016036987

Text:
ic requirements.) In the rare 
situation in which the employee who placed the lockout/tagout 
device is unable to remove that device, another person may 
remove it under the direction of the employer, provided that the 
employer strictly adheres to the specific procedures outlined in 
the standard. (See 29 CFR  1910.147(e)(3).) 
When do I use lockout and how do

In [43]:
retrieved_docs = results["documents"][0]
retrieved_meta = results["metadatas"][0]

In [44]:
context = ""

for i, (text, meta) in enumerate(
    zip(filtered_docs, filtered_meta),
    start=1
):
    context += f"""
[Source {i}]
File: {meta['source']}
Page: {meta['page']}

{text}

"""
if not filtered_docs:
    print("I could not find relevant information in the documents.")

In [31]:
prompt = f"""
You are answering questions using retrieved document context.

Use only the context below.

If the answer is not present, say:
"I could not find this information in the provided documents."

When answering, mention the source number in square brackets,
for example [Source 1].

Context:
{context}

Question:
{question}

Answer clearly and concisely.
"""

In [32]:
from ollama import chat

response = chat(
    model="qwen3:4b",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)
print("\nSources:")

for i, meta in enumerate(retrieved_meta, start=1):
    print(
        f"[Source {i}] "
        f"{meta['source']} - Page {meta['page']}"
    )

print(response.message.content)


Sources:
[Source 1] lockout_tagout.pdf - Page 19
[Source 2] lockout_tagout.pdf - Page 22
[Source 3] lockout_tagout.pdf - Page 15
Lockout/tagout is used to prevent exposure to hazardous energy during service and maintenance operations on machines [Source 3].


In [45]:
test_questions = [
    {
        "question": "What PPE is required in the production area?",
        "expected_source": "ppe_manual.pdf"
    },
    {
        "question": "What is lockout tagout used for?",
        "expected_source": "lockout_tagout.pdf"
    },
    {
        "question": "Why are machine guards important?",
        "expected_source": "machine_guarding.pdf"
    }
]

In [48]:
correct = 0

for item in test_questions:
    question = item["question"]

    question_embedding = embedding_model.encode(
        question,
        convert_to_numpy=True
    )

    results = collection.query(
        query_embeddings=[question_embedding.tolist()],
        n_results=3
    )

    retrieved_sources = [
        meta["source"]
        for meta in results["metadatas"][0]
    ]

    print("\nQuestion:", question)
    print("Expected:", item["expected_source"])
    print("Retrieved:", retrieved_sources)

    if item["expected_source"] in retrieved_sources:
        correct += 1


Question: What PPE is required in the production area?
Expected: ppe_manual.pdf
Retrieved: ['ppe_manual.pdf', 'ppe_manual.pdf', 'ppe_manual.pdf']

Question: What is lockout tagout used for?
Expected: lockout_tagout.pdf
Retrieved: ['lockout_tagout.pdf', 'lockout_tagout.pdf', 'lockout_tagout.pdf']

Question: Why are machine guards important?
Expected: machine_guarding.pdf
Retrieved: ['machine_guarding.pdf', 'machine_guarding.pdf', 'machine_guarding.pdf']


In [49]:
accuracy = (correct / len(test_questions)) * 100

print(f"\nCorrect retrievals: {correct}/{len(test_questions)}")
print(f"Retrieval Accuracy: {accuracy:.2f}%")


Correct retrievals: 3/3
Retrieval Accuracy: 100.00%
